In [9]:
from os import listdir
from os.path import isfile, join
import pandas as pd
import numpy as np
from os import listdir
from os.path import isfile, join
from CD_zoo.tools.scoring_tools import score
from sklearn.metrics import (
    roc_auc_score,
)
import pickle
from itertools import combinations

from os import listdir
from dl_components.linear import SimpleLinear
from dl_components.pl_wrappers import Architecture_PL
from os.path import isfile, join

from sklearn.metrics import (
    roc_auc_score,
)
import torch
import hydra
from omegaconf import OmegaConf
from hydra import compose
from lightning.pytorch.callbacks import ModelCheckpoint

In [10]:
# HEre we generate the basic ensemble strategies as well as check the individual performances of the methods on the new data.m

In [11]:
(X,X_names,method_order, training_labels, mapping) = pickle.load(open("/home/datasets4/stein/robust_exp/ensemble_experiment/trainsmall_.p", "rb"))
(X_eval,X_names_eval,method_order_eval, testing_labels, mapping_eval) = pickle.load(open("/home/datasets4/stein/robust_exp/ensemble_experiment/testsmall_.p", "rb"))

In [12]:
X[1].shape,training_labels[1].shape, len(training_labels)

((40, 7, 100, 5, 5, 3), (40, 100, 5, 5, 3), 27)

In [13]:
X_r = np.concatenate(X)
X_r = torch.tensor(X_r.transpose(0,2,3,4,5,1)).reshape(-1,7,7,4,7)

Y_r = np.concatenate(training_labels)
Y_r = torch.tensor(Y_r).reshape(-1, 7, 7, 4)

print(X_r.shape, Y_r.shape)

RuntimeError: shape '[-1, 7, 7, 4, 7]' is invalid for input of size 53550000

In [14]:
roc_auc_score((Y_r[20] != 0).flatten(), X_r[20,:,:,:,3].flatten())

NameError: name 'Y_r' is not defined

In [493]:
Y_r[3].shape

(100, 7, 7, 4)

In [ ]:
print(np.any([8 in x.shape for x in X]))

False


In [15]:
stack = []
for x in range(len(X)):
    stack.append((X_names[x]["ds"].unique()[0],roc_auc_score((training_labels[x].flatten() != 0),X[x].mean(axis=1).flatten())))
print(pd.DataFrame(stack).mean(numeric_only=True))

1    0.93369
dtype: float64


In [16]:
stack

[('instant_confounder_small', 0.9989836911371465),
 ('ino_shock_n_small', 0.9997639336351021),
 ('ino_time_n_small', 0.9996433711426017),
 ('lagged_confounder_small', 0.8593343811674199),
 ('obs_auto_n_small', 0.8721098542691492),
 ('standardization_small', 0.999541407254439),
 ('nl_power_small', 0.989491880836545),
 ('obs_mul_n_small', 0.8860218938074838),
 ('interpolate_nl_small', 0.7835163882095059),
 ('missing_info_small', 0.8043849428676677),
 ('obs_time_n_small', 0.8855461798561151),
 ('obs_shock_n_small', 0.9725680499090714),
 ('obs_common_n_small', 0.9397101605330784),
 ('ino_mul_n_small', 0.9988339872934399),
 ('faith_inst_small', 0.9958460451565609),
 ('length_small', 0.8490972390181631),
 ('rbf_nl_small', 0.889804460290339),
 ('faith_lagged_small', 0.9922509352904042),
 ('obs_add_n_small', 0.8870776699888909),
 ('ino_common_n_small', 0.9910264090975731),
 ('ino_uniform_n_small', 0.9998190796598627),
 ('symbolic_nl_small', 0.8274438448744709),
 ('splines_nl_small', 0.99593169

In [443]:
stack = []
for x in range(len(X)):
    stack.append((X_names[x]["ds"].unique()[0],roc_auc_score((training_labels[x].flatten() != 0),X[x].mean(axis=1).flatten())))
print(pd.DataFrame(stack).mean(numeric_only=True))

1    0.941366
dtype: float64


In [445]:
stack = []
for x in range(len(X_eval)):
    stack.append((X_names_eval[x]["ds"].unique()[0],roc_auc_score((testing_labels[x].flatten() != 0),X_eval[x].mean(axis=1).flatten())))
print(pd.DataFrame(stack).mean(numeric_only=True))

1    0.928051
dtype: float64


In [447]:
for i, item in enumerate(method_order):

    stack = []
    for x in range(len(X)):
        stack.append((X_names[x]["ds"].unique()[0],roc_auc_score((training_labels[x].flatten() != 0),X[x][:,i].flatten())))
    print(item,pd.DataFrame(stack).mean(numeric_only=True))

direct_crosscorr 1    0.835689
dtype: float64
dynotears 1    0.928391
dtype: float64
ntsnotears 1    0.920712
dtype: float64
pcmci 1    0.92519
dtype: float64
pcmciplus 1    0.927629
dtype: float64
var 1    0.932629
dtype: float64
varlingam 1    0.912653
dtype: float64


In [446]:
for i, item in enumerate(method_order):

    stack = []
    for x in range(len(X_eval)):
        stack.append((X_names_eval[x]["ds"].unique()[0],roc_auc_score((testing_labels[x].flatten() != 0),X_eval[x][:,i].flatten())))
    print(item,pd.DataFrame(stack).mean(numeric_only=True))

direct_crosscorr 1    0.81627
dtype: float64
dynotears 1    0.917599
dtype: float64
ntsnotears 1    0.89045
dtype: float64
pcmci 1    0.902594
dtype: float64
pcmciplus 1    0.904948
dtype: float64
var 1    0.920035
dtype: float64
varlingam 1    0.895774
dtype: float64


In [437]:
pd.DataFrame(stack).mean()

/tmp/ipykernel_294441/3093448272.py:1: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  pd.DataFrame(stack).mean()


1    0.928391
dtype: float64

In [418]:
X_names

[                     ds                                        dpath
 0   standardization_big  2025-06-18 19:34:14.6859_314.38909679319556
 1   standardization_big   2025-06-18 19:34:18.3344_210.6677879380982
 2   standardization_big   2025-06-18 19:34:21.5642_923.5875276023199
 3   standardization_big   2025-06-18 19:34:24.8982_607.8122781944489
 4   standardization_big    2025-06-18 19:34:28.2285_422.950128290474
 5   standardization_big   2025-06-18 19:34:37.1006_769.1450305164202
 6   standardization_big  2025-06-18 19:34:46.5014_470.53566229232734
 7   standardization_big  2025-06-18 19:34:55.3183_25.281327302455374
 8   standardization_big   2025-06-18 19:35:04.0948_912.8085377946007
 9   standardization_big  2025-06-18 19:35:12.8076_269.05527656140106
 10  standardization_big    2025-06-18 19:35:16.1334_536.934609395327
 11  standardization_big   2025-06-18 19:35:20.0953_629.3508127719142
 12  standardization_big   2025-06-18 19:35:24.5375_467.0624690191155
 13  standardization

In [401]:
(training_labels[0,0,].flatten() != 0 ).sum()

1407

In [ ]:
X_r = torch.tensor(X)
X_r = X_r.permute(0, 1, 3, 4,5,6,2)  # move dimension 3 to the last position to keep it cleaner.

In [375]:
X_r.shape

torch.Size([2, 40, 100, 7, 7, 4, 7])

In [379]:
X_r[:,:,:,0].shape

torch.Size([2, 40, 100, 7, 4, 7])

In [403]:
(training_labels[1].flatten() != 0).sum()

84675

In [404]:
roc_auc_score((training_labels[1].flatten() != 0),X_r[1,:,:,:,:,:,0].flatten())

0.8940666655373142

In [287]:
# Data loader transform to match the order of the methods correctly.
X_r = X.reshape(-1,X.shape[2],7,7,4)
X_r = X_r.reshape(X_r.shape[0],7,-1,4)
X_r = torch.tensor(X_r, dtype=torch.float32)

Y_r = training_labels.reshape(-1,7,7,4)


a = torch.tensor(X.reshape(X.shape[0],X.shape[1],100,7,49,4), dtype=torch.float32)


print(np.all(Y_r[0] == training_labels[0][0][0]))
print(torch.all(X_r[0] == a[0][0][0]))

True
tensor(True)


In [ ]:
# Transform should match here.

print(np.all(Y_r[:100] == training_labels[0][0]))
print(torch.all(X_r[:100][:,:,:7] == X[0][0][0]))

True


tensor(False)

In [345]:
X_r[:100][:,:,:7].shape

torch.Size([100, 7, 7, 4])

In [342]:
X_r[:100].shape

torch.Size([100, 7, 49, 4])

In [338]:
roc_auc_score((Y_r[0].flatten() != 0),X_r[0][:,:7].flatten())

0.5314917127071823

In [337]:
Y_r[0].shape

(7, 7, 4)

In [324]:
roc_auc_score((training_labels[0,0,].flatten() != 0),X[0,0,0].flatten())

0.9851293977302751

In [302]:
def calc_auroc_per_ds_for_violation(model,X,training_labels, use_model=True):
    pred_stack = []
    for x in range(training_labels.shape[0]):
        if use_model:
            pred = model.model(torch.Tensor(X[x].swapaxes(0,1).reshape(100,7,49,4))).detach().numpy()
        else:
            pred = X[x][:,:,:7,:]
        pred_stack.append(roc_auc_score(
            (training_labels[x] !=0).flatten(),
            pred.flatten(),
        ))
    return pred_stack

In [303]:
model_path = "/home/datasets4/stein/robust_exp/ensemble_experiment/deep_runs/2025-07-25_14-27-50-476147/last.ckpt"
model = Architecture_PL.load_from_checkpoint(model_path).to('cpu')

/home/stein/anaconda3/envs/cd_zoo_base/lib/python3.9/site-packages/lightning/pytorch/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.2, which is newer than your current Lightning version: v2.5.0.post0


In [304]:
res = np.mean(calc_auroc_per_ds_for_violation(model,a[0],training_labels[0],use_model=False))

In [306]:
res

0.5033645962298386

In [301]:
a[0].shape

torch.Size([40, 100, 7, 49, 4])

In [ ]:
roc_auc_score(
            (training_labels[x] !=0).flatten(),

In [ ]:
res = np.mean(calc_auroc_per_ds_for_violation(model,a[0],training_labels[0]))

In [125]:
LinearModel = SimpleLinear.load_from_checkpoint(model_path, input_dim=X.shape[1], output_dim=1)

AttributeError: type object 'SimpleLinear' has no attribute 'load_from_checkpoint'

In [ ]:
tr_auroc = pd.read_csv("/home/datasets4/stein/robust_exp/ensemble_experiment/export_train/mean/correct_lag/WCG/AUROC Joint/highest_hps.csv", index_col=0)
eval_auroc = pd.read_csv("/home/datasets4/stein/robust_exp/ensemble_experiment/export_eval/mean/correct_lag/WCG/AUROC Joint/highest_hps.csv", index_col=0)
tr_auroc2 = pd.read_csv("/home/datasets4/stein/robust_exp/main_experiment/export/mean/correct_lag/WCG/AUROC Joint/highest_hps.csv", index_col=0)

In [ ]:
1. get wrongly generated stuff
2. generate summarize etc if its done

3. CHECK consistency
4. Normalize probability space
5. Train and pray. 



In [54]:
tr_auroc2.max(axis=1).mean()

0.9346929459599543

In [64]:
X[0][:,0][0].shape

(100, 7, 7, 4)

In [76]:
X.shape

(2, 40, 7, 100, 7, 7, 4)

In [79]:
auroc_stack = []
for x in range(40):
    auroc_stack.append(roc_auc_score(training_labels[0][x].flatten() != 0,X[0][x][0].flatten()))




In [81]:
np.mean(auroc_stack)

0.8986951062933418

In [82]:
tr_auroc

,cp,pcmci,ntsnotears,pcmciplus,varlingam,dynotears,var,direct_crosscorr
obs_common_n,0.840428,0.839478,0.890962,0.800911,0.899796,0.926059,0.881537,0.816594
ino_shock_n,0.951412,0.964328,0.952836,0.963886,0.972119,0.986855,0.987877,0.896043
nonstat_n,0.953879,0.997945,0.997199,0.997148,0.998704,0.999759,0.999088,0.931643
obs_shock_n,0.839155,0.940705,0.833280,0.942269,0.803635,0.852536,0.849860,0.820694
standardization,0.956273,0.998089,0.997317,0.997200,0.998872,0.995687,0.998729,0.935464
splines_nl,0.975158,0.989912,0.990937,0.990933,0.981403,0.994403,0.993173,0.982300
faith_lagged,0.847397,0.984908,0.967633,0.984272,0.977478,0.961244,0.990387,0.846890
obs_mul_n,0.838594,0.839891,0.826991,0.842637,0.800248,0.853861,0.846255,0.815384
obs_add_n,0.837331,0.835866,0.827224,0.839878,0.799434,0.858091,0.846565,0.817610
interpolate_nl,0.701006,0.776395,0.754043,0.784553,0.712760,0.731403,0.766371,0.705517


In [47]:
eval_auroc.mean()

cp                  0.865681
pcmci               0.931004
ntsnotears          0.926939
pcmciplus           0.928826
varlingam           0.912477
dynotears           0.914655
var                 0.933588
direct_crosscorr    0.872383
dtype: float64

In [48]:
tr_auroc2.mean()

cp                  0.864601
pcmci               0.923053
ntsnotears          0.916788
pcmciplus           0.921840
varlingam           0.927879
dynotears           0.911023
var                 0.926592
direct_crosscorr    0.862467
dtype: float64

In [ ]:
auroc = roc_auc_score(labels_eval[mapping].flatten() != 0, res[mapping].flatten())


In [26]:
# Load hydra config
from omegaconf import OmegaConf
cfg = OmegaConf.load("config/7_baseline_ensemble_performance.yaml")

In [27]:
X_tr,X_names_tr,method_order_tr, Y_tr, Y_names_tr = pickle.load(open(cfg.path + cfg.tr_data_path, "rb"))
X_te,X_names_te,method_order_te, Y_te, Y_names_te = pickle.load(open(cfg.path + cfg.te_data_path, "rb"))


print(X_tr.shape)
print(Y_tr.shape)
print("Done. Check csv logs.")


FileNotFoundError: [Errno 2] No such file or directory: '/home/datasets4/stein/robust_exp/ensemble_experiment/train.p'

In [28]:
def load_full_ds(mypath="training_ensemble/", restrict=None):
    onlyfiles = [f for f in listdir(mypath) if not isfile(join(mypath, f))]
    if restrict is not None:
        onlyfiles = [f for f in onlyfiles if f in restrict]

    label_stack = []
    naming_stack = []

    for x in onlyfiles:
        runs = [
            mypath + x + "/" + f
            for f in listdir(mypath + x)
            if not isfile(join(mypath, f))
        ]
        for item in runs:
            label = np.load(item + "/Y.npy")
            label_stack.append(label)
            naming_stack.append([mypath, item.split("/")[-2], item.split("/")[-1]])
    return np.stack(label_stack), pd.DataFrame(
        naming_stack, columns=["ds", "violation", "run"]
    )

## load cached

In [29]:
def get_predictions_for_individual_ds(
    mapping, predictions, naming, method_restriction=None
):
    """
    Get predictions for a specific dataset.
    """
    naming = (
        naming[naming["model"].isin(method_restriction)]
        if method_restriction is not None
        else naming
    )

    stack = []
    namings = []
    for run in mapping["run"].unique():
        a = naming[naming["dpath"] == run]
        stack.append(predictions[a.index].mean(axis=0))
        namings.append(run)
    return np.stack(stack), pd.DataFrame(namings, columns=["run"])

In [30]:
def test_case(base_results, mapping_eval, predictions_eval, naming_eval):
    # THis should match.
    # test to check if the metrics is matching with previously calculated: 
    test_case = base_results[base_results["method.name"] == "var"]
    test_paths = [x.split("/")[-1] for x in test_case["path"]]

    res, res_naming = get_predictions_for_individual_ds(
        mapping_eval, predictions_eval, naming_eval, method_restriction=["var"]
    )

    stack = []
    for x in test_paths:
        mapping = res_naming[x == res_naming.run.values].index.values[0]
        auroc = roc_auc_score(labels_eval[mapping].flatten() != 0, res[mapping].flatten())
        stack.append(auroc)

    assert np.max(np.abs(stack - test_case["AUROC Joint"].values)) < 1e-5, "Missmatch in AUROC values"

In [31]:
labels, mapping = load_full_ds()
labels_eval, mapping_eval = load_full_ds(
    "data/", restrict=mapping["violation"].unique()
)
predictions_eval, naming_eval = pickle.load(
    open("ensemble_experiment/predictions_eval.p", "rb")
)
# Everything should match here:
[x for x in naming_eval["dpath"] if x not in mapping_eval["run"].values]

FileNotFoundError: [Errno 2] No such file or directory: 'training_ensemble/'

In [21]:
# Remove non WCG
naming_eval = naming_eval[
    ~naming_eval.model.isin(["crosscorr", "physical", "combo", "cp"])
]
predictions_eval = np.array(
    [
        predictions_eval[x]
        for x in range(len(predictions_eval))
        if (x in naming_eval.index)
    ]
)
naming_eval.reset_index(drop=True, inplace=True)

NameError: name 'naming_eval' is not defined

In [34]:
# Load previously calulated:
mypath = "/home/datasets4/stein/robust_exp/ensemble_experiment/ensemble_summarized_results_eval/WCG/"
onlyfiles = [mypath + f for f in listdir(mypath) if isfile(join(mypath, f))]
base_results = pd.concat([pd.read_csv(x,index_col=0) for x in onlyfiles])
base_results = base_results[~(base_results["method.name"] == "cp")]

FileNotFoundError: [Errno 2] No such file or directory: '/home/datasets4/stein/robust_exp/ensemble_experiment/ensemble_summarized_results_eval/WCG/'

In [10]:
# should match
print([x for x in naming_eval["dpath"] if x not in mapping_eval["run"].values])
paths = [x.split("/")[-1] for x in base_results["path"]]
print([x for x in naming_eval["dpath"] if x not in paths])

[]
[]


In [11]:
# Run test to see if scoring matches: 
test_case(base_results, mapping_eval, predictions_eval, naming_eval)

In [13]:
# Base performance for each method: 
base_performance = base_results[["method.name", "AUROC Joint"]].groupby("method.name").mean()
latex_table = base_performance.to_latex()
print(latex_table)
base_performance

\begin{tabular}{lr}
\toprule
{} &  AUROC Joint \\
method.name      &              \\
\midrule
direct\_crosscorr &     0.763022 \\
dynotears        &     0.835603 \\
ntsnotears       &     0.798055 \\
pcmci            &     0.796649 \\
pcmciplus        &     0.785167 \\
var              &     0.810210 \\
varlingam        &     0.786131 \\
\bottomrule
\end{tabular}



/tmp/ipykernel_368494/1843506433.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex_table = base_performance.to_latex()


,AUROC Joint
method.name,
direct_crosscorr,0.763022
dynotears,0.835603
ntsnotears,0.798055
pcmci,0.796649
pcmciplus,0.785167
var,0.810210
varlingam,0.786131


In [294]:
# Pareto aggregation: 
base_results[["path", "AUROC Joint"]].groupby("path").max().mean()

AUROC Joint    0.839007
dtype: float64

In [299]:
# Pareto front is smaller than the full set: 
for x in base_results["method.name"].unique():
    separation = [x]
    a = base_results[base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
    b = base_results[~base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
    change = ((a > b ).sum() / len(b)).values[0]
    print(x, change)

ntsnotears 0.0125
pcmciplus 0.020833333333333332
varlingam 0.041666666666666664
direct_crosscorr 0.12916666666666668
dynotears 0.7208333333333333
var 0.05
pcmci 0.020833333333333332


In [360]:
# Where does dynotears fail? 
separation = ["dynotears"]
a = base_results[base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
b = base_results[~base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
change = ((a > b ).sum() / len(b)).values[0]
failures = base_results[base_results["path"].isin(a[(a < b).values].index)]
failures = failures[failures["method.name"] == "dynotears"]
# Where does dynotears fail? 
variations_columns = ["generator.time_series_n", "generator.struc.link_proba", "generator.instant.link_proba","generator.obs_n.snr"]
for x in variations_columns:
    print(failures[x].value_counts())

250     35
1000    32
Name: generator.time_series_n, dtype: int64
0.150    34
0.075    33
Name: generator.struc.link_proba, dtype: int64
0.0    37
0.1    30
Name: generator.instant.link_proba, dtype: int64
0.1     31
0.5     25
1.0      6
10.0     3
5.0      2
Name: generator.obs_n.snr, dtype: int64


In [365]:
# What is the best method for the low SNR regime? 
low_regime = base_results[base_results["generator.obs_n.snr"].isin([0,.1,0.5])]
# Base performance for each method: 
low_regime = low_regime[["method.name", "AUROC Joint"]].groupby("method.name").mean()
low_regime

,AUROC Joint
method.name,
direct_crosscorr,0.672781
dynotears,0.715454
ntsnotears,0.684859
pcmci,0.675988
pcmciplus,0.674193
var,0.696098
varlingam,0.648071


In [387]:
# Is there a method that performs best in the cases where dynotears fails? 
separation = ["dynotears"]
a = base_results[base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
b = base_results[~base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
change = ((a > b ).sum() / len(b)).values[0]
fail_regime = base_results[base_results["path"].isin(a[(a < b).values].index)]
for x in base_results["method.name"].unique():
    separation = [x]
    a = fail_regime[fail_regime["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
    b = fail_regime[~fail_regime["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
    change = ((a > b ).sum() / len(b)).values[0]
    print(x, change)

ntsnotears 0.04477611940298507
pcmciplus 0.07462686567164178
varlingam 0.14925373134328357
direct_crosscorr 0.4626865671641791
dynotears 0.0
var 0.1791044776119403
pcmci 0.07462686567164178


In [ ]:
# Is there a combo of two that is mostly better? 
combo_stack = []
for x in combinations(base_results["method.name"].unique(), 2):
    separation = x
    a = base_results[base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
    b = base_results[~base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
    change = ((a > b ).sum() / len(b)).values[0]
    combo_stack.append((x, change))
pd.DataFrame(combo_stack, columns=["methods", "change"]).sort_values("change", ascending=False)

,methods,change
15,"(direct_crosscorr, dynotears)",0.850000
18,"(dynotears, var)",0.770833
12,"(varlingam, dynotears)",0.762500
19,"(dynotears, pcmci)",0.741667
8,"(pcmciplus, dynotears)",0.741667
3,"(ntsnotears, dynotears)",0.733333
16,"(direct_crosscorr, var)",0.179167
11,"(varlingam, direct_crosscorr)",0.170833
17,"(direct_crosscorr, pcmci)",0.150000
7,"(pcmciplus, direct_crosscorr)",0.150000


In [388]:
# Is there a combo of 3 that is mostly better? 
combo_stack = []
for x in combinations(base_results["method.name"].unique(), 3):
    separation = x
    a = base_results[base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
    b = base_results[~base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
    change = ((a > b ).sum() / len(b)).values[0]
    combo_stack.append((x, change))
pd.DataFrame(combo_stack, columns=["methods", "change"]).sort_values("change", ascending=False)[:10]

,methods,change
31,"(direct_crosscorr, dynotears, var)",0.900000
25,"(varlingam, direct_crosscorr, dynotears)",0.891667
19,"(pcmciplus, direct_crosscorr, dynotears)",0.870833
32,"(direct_crosscorr, dynotears, pcmci)",0.870833
9,"(ntsnotears, direct_crosscorr, dynotears)",0.862500
28,"(varlingam, dynotears, var)",0.812500
34,"(dynotears, var, pcmci)",0.795833
22,"(pcmciplus, dynotears, var)",0.791667
16,"(pcmciplus, varlingam, dynotears)",0.783333
12,"(ntsnotears, dynotears, var)",0.783333


In [390]:
# Is there a combo of 3 that is mostly better? 
combo_stack = []
for x in combinations(base_results["method.name"].unique(), 4):
    separation = x
    a = base_results[base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
    b = base_results[~base_results["method.name"].isin(separation)][["path", "AUROC Joint"]].groupby("path").max()
    change = ((a > b ).sum() / len(b)).values[0]
    combo_stack.append((x, change))
pd.DataFrame(combo_stack, columns=["methods", "change"]).sort_values("change", ascending=False)[:10]

,methods,change
30,"(varlingam, direct_crosscorr, dynotears, var)",0.941667
34,"(direct_crosscorr, dynotears, var, pcmci)",0.925000
26,"(pcmciplus, direct_crosscorr, dynotears, var)",0.920833
31,"(varlingam, direct_crosscorr, dynotears, pcmci)",0.912500
20,"(pcmciplus, varlingam, direct_crosscorr, dynot...",0.912500
16,"(ntsnotears, direct_crosscorr, dynotears, var)",0.912500
10,"(ntsnotears, varlingam, direct_crosscorr, dyno...",0.904167
27,"(pcmciplus, direct_crosscorr, dynotears, pcmci)",0.891667
17,"(ntsnotears, direct_crosscorr, dynotears, pcmci)",0.883333
4,"(ntsnotears, pcmciplus, direct_crosscorr, dyno...",0.883333


In [305]:
res, res_naming = get_predictions_for_individual_ds(
    mapping_eval, predictions_eval, naming_eval, method_restriction=["dynotears","direct_crosscorr"]
)
stack = []
for x in paths:
    mapping = res_naming[x == res_naming.run.values].index.values[0]
    auroc = roc_auc_score(labels_eval[mapping].flatten() != 0, res[mapping].flatten())
    stack.append(auroc)
# THis should match.
np.mean(stack)

0.8128967859272941

In [389]:
res, res_naming = get_predictions_for_individual_ds(
    mapping_eval, predictions_eval, naming_eval, method_restriction=["dynotears","direct_crosscorr", "var"]
)
stack = []
for x in paths:
    mapping = res_naming[x == res_naming.run.values].index.values[0]
    auroc = roc_auc_score(labels_eval[mapping].flatten() != 0, res[mapping].flatten())
    stack.append(auroc)
# THis should match.
np.mean(stack)

0.8204393537139847

In [391]:
res, res_naming = get_predictions_for_individual_ds(
    mapping_eval, predictions_eval, naming_eval, method_restriction=["dynotears","direct_crosscorr", "var", "varlingam"]
)
stack = []
for x in paths:
    mapping = res_naming[x == res_naming.run.values].index.values[0]
    auroc = roc_auc_score(labels_eval[mapping].flatten() != 0, res[mapping].flatten())
    stack.append(auroc)
# THis should match.
np.mean(stack)

0.8282919149474629

In [392]:
res, res_naming = get_predictions_for_individual_ds(
    mapping_eval, predictions_eval, naming_eval, method_restriction=None
)
stack = []
for x in paths:
    mapping = res_naming[x == res_naming.run.values].index.values[0]
    auroc = roc_auc_score(labels_eval[mapping].flatten() != 0, res[mapping].flatten())
    stack.append(auroc)
# THis should match.
np.mean(stack)

0.8201944179921765